# Lab 18: The Colour Teleporter

*Brain Lab: 84 Experiments to Decode Your Biological and Artificial Intelligence* · Chapter 2, *The Time Editor*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/labphonlab/brain-lab-support/blob/main/notebooks/lab-18.ipynb)

**What you will do:** watch the colour phi phenomenon turn two flashing dots into one moving, colour-changing dot, log where along the path the colour seems to switch, and interpolate between two words in embedding space to see a language model fill in a comparable "in-between" state.

**How to use.** Run the cells from top to bottom (Shift+Enter). Nothing needs installing, and nothing you do here can break anything: *Runtime → Restart and run all* returns it to the start. To keep your own results, use *File → Save a copy in Drive*.

In [ ]:
# Setup: run this cell first (Shift+Enter).
import platform, numpy as np, pandas as pd, matplotlib
import matplotlib.pyplot as plt
np.random.seed(2026)
print("Python", platform.python_version(), "| numpy", np.__version__, "| pandas", pd.__version__)

## The experiment in the book

In Lab 18 you watched a red dot flash on the left and a green dot flash on the right,
alternating every 100-200 ms. Instead of two flashing dots, you saw a single dot sweep
smoothly from left to right, switching colour partway along its invisible path, at a location
where nothing was ever shown. This is the **colour phi phenomenon**
(Kolers & von Grünau, 1976): the brain waits for both flashes, then constructs a coherent
motion-and-colour narrative and presents the whole thing to consciousness as if it had
actually happened that way.

## Record your results

Using the online demonstration, vary the interval between flashes (if the demo allows it)
and, for each setting, rate two things: how convincing the apparent motion is (0-5) and
roughly where along the path the colour seems to switch, as a fraction from 0 (still red, at
the left dot) to 1 (still green, at the right dot).

In [ ]:
# example data: replace with your own settings and ratings
trials = pd.DataFrame({
    "interval_ms": [80, 120, 160, 200, 250],
    "motion_convincing": [2, 4, 5, 4, 2],
    "colour_switch_fraction": [np.nan, 0.55, 0.5, 0.45, np.nan],  # NaN if motion wasn't seen at all
})
display(trials)
trials.plot(x="interval_ms", y="motion_convincing", marker="o", color="steelblue", figsize=(6, 3))
plt.ylabel("motion convincingness (0-5)"); plt.xlabel("interval between flashes (ms)"); plt.show()

## Compare

Lab 18 gives a range for the flash interval that produces the effect, "roughly 100-200
milliseconds per flash," and describes the colour switch as happening "at the midpoint of
its apparent path," i.e. a fraction near 0.5. Check both against your own data.

In [ ]:
in_range = trials["interval_ms"].between(100, 200)
print("Intervals in the lab's 100-200 ms range:", trials.loc[in_range, "interval_ms"].tolist())
print("Your motion ratings at those intervals:", trials.loc[in_range, "motion_convincing"].tolist())

near_midpoint = trials["colour_switch_fraction"].dropna()
if len(near_midpoint):
    mean_fraction = near_midpoint.mean()
    print(f"\nMean colour-switch fraction (0 = left dot, 1 = right dot): {mean_fraction:.2f}")
    if 0.4 <= mean_fraction <= 0.6:
        print("Close to the midpoint (0.5), as the lab describes.")
    else:
        print("Some way from the midpoint. This can happen if the two flashes are not equally salient"
              " (e.g. the display's red and green are not equally bright), which can pull the perceived"
              " switch point towards the more dominant colour.")
else:
    print("\nMotion was not convincing enough at any interval to judge a colour-switch point -- try")
    print("intervals closer to the middle of the 100-200 ms range.")

## The AI side

The lab's AI Connection compares the colour phi effect to **latent space interpolation**:
moving smoothly between two points in a model's internal representation generates plausible
in-between states that were never in the training data. Word embeddings such as GloVe
(Pennington, Socher & Manning, 2014, EMNLP) place words with similar meanings near each other
in a high-dimensional space. Below, we interpolate in a straight line between the embeddings
for two words and find the real words nearest to each intermediate point along the way, a
linguistic version of the "filled-in middle" you saw between the red and green dots.

*This part downloads a small open model or library, so it runs in Colab (it may take a minute the first time).*

In [ ]:
import gensim.downloader as api

vectors = api.load("glove-wiki-gigaword-50")   # small (~66 MB), downloads once

word_a, word_b = "king", "cold"
n_steps = 6
vec_a, vec_b = vectors[word_a], vectors[word_b]

rows = []
for i in range(n_steps + 1):
    t = i / n_steps
    mixed = (1 - t) * vec_a + t * vec_b
    nearest = vectors.similar_by_vector(mixed, topn=1)[0][0]
    rows.append({"t (0=word_a, 1=word_b)": round(t, 2), "nearest real word": nearest})
path = pd.DataFrame(rows)
display(path)
print(f"Interpolating from {word_a!r} to {word_b!r}: " + " -> ".join(path["nearest real word"]))

Somewhere along the path, the nearest real word usually shifts abruptly from words related to
word_a to words related to word_b, often via an unrelated word in between; there is no
smooth, meaningful "half king, half cold" word waiting in the space, even though the
arithmetic midpoint is a perfectly well-defined vector. That is the disanalogy with the
colour phi effect: your visual system fills the gap with a plausible in-between percept (a
colour partway between red and green would even be plausible chromatically), while the
embedding space's geometric midpoint often lands nowhere near a plausible in-between word,
because word meanings are not smoothly continuous the way colour and position are.

## Pool the class data

Pool the class's reports of where the colour switch seemed to happen (Step 3), as a fraction
of the path from the first dot (0) to the second (1).

In [ ]:
import io
csv_text = """id,colour_switch_fraction
P01,0.5
P02,0.45
P03,0.55
P04,0.5
P05,0.6
P06,0.4
"""  # example data: replace with your class CSV, e.g. pd.read_csv("colourphi_class.csv")
class_df = pd.read_csv(io.StringIO(csv_text))
boot = [class_df["colour_switch_fraction"].sample(len(class_df), replace=True).mean() for _ in range(5000)]
lo, hi = np.percentile(boot, [2.5, 97.5])
print(f"Class mean switch point: {class_df['colour_switch_fraction'].mean():.2f} "
      f"(95% bootstrap interval {lo:.2f}-{hi:.2f}, n = {len(class_df)}; midpoint = 0.5)")
class_df["colour_switch_fraction"].plot.hist(bins=6, color="steelblue", alpha=0.7, figsize=(6, 3))
plt.axvline(0.5, color="grey", linestyle="--", label="midpoint"); plt.legend(); plt.show()

## Questions to think about

1. The lab poses this question directly -- how does the brain know to change the colour at the midpoint before the second dot has even appeared? In your own words, explain why this requires the brain to wait, rather than commit to a perception the instant the first dot flashes.
2. In the embedding interpolation demo, is there ever a step where the nearest word is genuinely "in between" word_a and word_b in meaning, or does it jump abruptly? What does this suggest about the difference between a continuous perceptual space (colour, position) and a space of discrete word meanings?
3. The AI Connection compares this to a diffusion model or VAE morphing a cat into a dog through intermediate images "that were never in its training data." Is the intermediate colour you perceive in the colour phi effect similarly "not in the training data," that is, not physically presented on the screen? Where exactly does the analogy hold, and where does it break down?
4. Try changing word_a and word_b in the code to two words you think are conceptually closer, e.g. "king" and "queen." Do you expect the interpolation path to pass through fewer, more clearly related words than the king-to-cold path? Run it and check.

## Challenge

Find or build a version of the colour phi demo where you can change the interval between
flashes. Starting well outside the 100-200 ms range described in the lab (e.g. 500 ms, where
you should just see two separate flashing dots) and moving into the range, log the interval at
which apparent motion first appears, and the interval at which it stops being convincing again
at the other end. Report the range you found and compare it with the lab's 100-200 ms.

## Record what you ran

In [ ]:
NOTEBOOK_ID = "lab-18.ipynb"
# Reproducibility record: paste this output into your notes or report.
import sys, platform, datetime, numpy, pandas, matplotlib
print("Notebook:", NOTEBOOK_ID)
print("Run at  :", datetime.datetime.now().isoformat(timespec="seconds"))
print("Python  :", sys.version.split()[0], "on", platform.platform())
print("numpy", numpy.__version__, "| pandas", pandas.__version__, "| matplotlib", matplotlib.__version__)
print("Seed    : 2026")